In [1]:
import sys
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from transformers import AutoTokenizer

sys.path.append("../")

# 1. Load the split datasets
print("-> Loading processed CSV splits...")
X_train = pd.read_csv("../data/processed/X_train.csv")["cleaned_text"].fillna("").tolist()
X_test = pd.read_csv("../data/processed/X_test.csv")["cleaned_text"].fillna("").tolist()
y_train = pd.read_csv("../data/processed/y_train.csv")["sentiment_label"].tolist()
y_test = pd.read_csv("../data/processed/y_test.csv")["sentiment_label"].tolist()
print(f"   Successfully loaded {len(X_train)} train and {len(X_test)} test rows.")

# 2. Instantiate the tokenizer
MODEL_CKPT = "distilbert-base-uncased"
print(f"-> Contacting Hugging Face to fetch '{MODEL_CKPT}' tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
print("   Tokenizer loaded successfully!")

# 3. Create PyTorch Dataset
class MovieReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = MovieReviewDataset(X_train, y_train, tokenizer)
test_dataset = MovieReviewDataset(X_test, y_test, tokenizer)

print("\nPyTorch Tokenization Datasets successfully built!")
sample_item = train_dataset[0]
print("Sample Tensor Shape (input_ids):", sample_item["input_ids"].shape)

-> Loading processed CSV splits...
   Successfully loaded 1200 train and 300 test rows.
-> Contacting Hugging Face to fetch 'distilbert-base-uncased' tokenizer...
   Tokenizer loaded successfully!

PyTorch Tokenization Datasets successfully built!
Sample Tensor Shape (input_ids): torch.Size([128])


In [2]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Define evaluation metrics using scikit-learn for stability
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

# 2. Load pre-trained DistilBERT weights configured for 3 classes
print("-> Loading pre-trained DistilBERT architecture for 3-class classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, 
    num_labels=3
)

# 3. Configure training arguments optimized for a clean, swift execution
training_args = TrainingArguments(
    output_dir="../models/transformer_checkpoints",
    eval_strategy="epoch",          # Run evaluation at the end of every epoch
    save_strategy="epoch",          # Keep checkpoints aligned with evaluation steps
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,             # 3 epochs is perfect for this corpus size
    weight_decay=0.01,
    load_best_model_at_end=True,    # Automatically roll back to the absolute best weights
    metric_for_best_model="f1_macro",
    logging_steps=20,
    report_to="none"                # Silences third-party dashboard connection warnings
)

# 4. Initialize the unified Hugging Face Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("\nTrainer successfully initialized and ready for fine-tuning!")

-> Loading pre-trained DistilBERT architecture for 3-class classification...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Trainer successfully initialized and ready for fine-tuning!


In [3]:
print("Starting 3-class DistilBERT fine-tuning loop...")
print("Training progress logs will update below at every epoch checkpoint.\n")

# Fire up the training engine!
train_results = trainer.train()

print("\n--- Training Completed Successfully! ---")
print(f"Total Global Training Steps: {train_results.global_step}")
print(f"Total Training Loss: {train_results.training_loss:.4f}")

Starting 3-class DistilBERT fine-tuning loop...
Training progress logs will update below at every epoch checkpoint.



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.095884,0.021647,1.000000,1.000000
2,0.012478,0.008173,1.000000,1.000000
3,0.008891,0.006557,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Training Completed Successfully! ---
Total Global Training Steps: 225
Total Training Loss: 0.1370


In [4]:
# Define the permanent local storage directory
model_dir = "../models/transformer_3class"

# Save the fine-tuned weights and the tokenizer configs
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

print(f"--- Export Complete ---")
print(f"Engine 3 multi-class transformer successfully saved to: {model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- Export Complete ---
Engine 3 multi-class transformer successfully saved to: ../models/transformer_3class


In [5]:
from transformers import pipeline

# Load the model directly from your newly created local folder
test_pipe = pipeline("sentiment-analysis", model="../models/transformer_3class", tokenizer="../models/transformer_3class")

# Test phrase with subtle neutral nuance
test_phrase = "The movie was mediocre. Fine acting but a highly standard plot."
result = test_pipe(test_phrase)[0]

# Class map tracking: LABEL_0=Negative, LABEL_1=Neutral, LABEL_2=Positive
class_labels = {"LABEL_0": "Negative", "LABEL_1": "Neutral", "LABEL_2": "Positive"}

print(f"Test Text: '{test_phrase}'")
print(f"Predicted Raw Tag: {result['label']} -> Resolved Class: {class_labels[result['label']]}")
print(f"Model Confidence Score: {result['score']:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test Text: 'The movie was mediocre. Fine acting but a highly standard plot.'
Predicted Raw Tag: LABEL_1 -> Resolved Class: Neutral
Model Confidence Score: 0.6336


In [1]:
import torch
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

# 1. Verify your local GPU is connected and ready
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device being used: {device.upper()}")
if device == "cuda":
    print(f"GPU Model detected: {torch.cuda.get_device_name(0)}")

# 2. Fetch the 355-million parameter weights and tokenizer
print("\nDownloading and loading RoBERTa-Large base weights...")
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)

# 3. Allocate the model to your GPU VRAM
print("Moving RoBERTa-Large weights to GPU memory...")
model.to(device)

print("\nSuccess! RoBERTa-Large is successfully initialized and active in VRAM.")

Device being used: CUDA
GPU Model detected: NVIDIA GeForce RTX 4050 Laptop GPU



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Moving RoBERTa-Large weights to GPU memory...

Success! RoBERTa-Large is successfully initialized and active in VRAM.


In [4]:
import os
import pandas as pd

# 1. Define the exact absolute paths using Windows raw strings
BASE_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed"

X_TRAIN_PATH = os.path.join(BASE_DIR, "X_train.csv")
Y_TRAIN_PATH = os.path.join(BASE_DIR, "y_train.csv")
X_VAL_PATH = os.path.join(BASE_DIR, "X_test.csv")
Y_VAL_PATH = os.path.join(BASE_DIR, "y_test.csv")

print("Loading existing data splits from processed directory...")

# 2. Read the CSV files
X_train = pd.read_csv(X_TRAIN_PATH).iloc[:, -1].astype(str).tolist()
y_train = pd.read_csv(Y_TRAIN_PATH).iloc[:, -1].tolist()
X_val = pd.read_csv(X_VAL_PATH).iloc[:, -1].astype(str).tolist()
y_val = pd.read_csv(Y_VAL_PATH).iloc[:, -1].tolist()

print(f"Successfully loaded {len(X_train)} training samples and {len(X_val)} validation samples.")

# 3. Tokenize the text sequences for RoBERTa-Large
print("\nTokenizing dataset text splits (this will take a moment with RoBERTa-Large)...")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=256)

print("Tokenization complete! Texts successfully converted to token matrices.")

Loading existing data splits from processed directory...
Successfully loaded 1200 training samples and 300 validation samples.

Tokenizing dataset text splits (this will take a moment with RoBERTa-Large)...
Tokenization complete! Texts successfully converted to token matrices.


In [5]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Create a custom PyTorch Dataset subclass
class MovieReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Convert dictionary list vectors into PyTorch Tensors on the fly
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 2. Wrap our tokenized train and validation splits
train_dataset = MovieReviewDataset(train_encodings, y_train)
val_dataset = MovieReviewDataset(val_encodings, y_val)

# 3. Define the evaluation matrix function for tracking model health
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # Calculate performance metrics across our 3 classes
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("PyTorch Datasets generated and tracking metrics successfully configured!")

PyTorch Datasets generated and tracking metrics successfully configured!


In [7]:
from transformers import TrainingArguments, Trainer

# 1. Define paths for saving the model and logs on Windows
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_large_3class"
LOGS_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\logs\roberta_training"

# 2. Set up optimized training configurations to protect your VRAM
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,                 # 3 epochs is standard for fine-tuning transformers
    per_device_train_batch_size=4,      # Low batch size prevents VRAM crashes
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # Simulates a batch size of 16 for stability
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=10,
    eval_strategy="epoch",              # Fixed: Updated from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                          # Uses mixed precision to accelerate your GPU
    report_to="none"
)

# 3. Initialize the Trainer execution engine
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 4. Kick off the fine-tuning loop
print("Starting RoBERTa-Large fine-tuning loop on your GPU...")
trainer.train()

# 5. Save the final optimized weights and tokenizer
print("\nTraining complete! Saving fine-tuned assets...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"All assets successfully saved to {OUTPUT_DIR}")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting RoBERTa-Large fine-tuning loop on your GPU...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.004492,0.000041,1.000000,1.000000,1.000000,1.000000
2,0.000055,0.000006,1.000000,1.000000,1.000000,1.000000
3,0.000048,0.000006,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete! Saving fine-tuned assets...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All assets successfully saved to F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_large_3class


In [8]:
import pandas as pd
from collections import Counter

# 1. Print out a few raw samples side-by-side
print("--- RAW DATA SAMPLES ---")
for i in range(5):
    print(f"\n[Sample #{i+1}]")
    print(f"Label: {y_train[i]}")
    print(f"Text: {X_train[i][:300]}...") # Shows the first 300 characters

print("\n" + "="*50 + "\n")

# 2. Check for exact duplicate reviews in the training set
print("--- DUPLICATE ANALYSIS ---")
train_series = pd.Series(X_train)
total_samples = len(train_series)
unique_samples = train_series.nunique()
duplicate_count = total_samples - unique_samples

print(f"Total training samples: {total_samples}")
print(f"Unique text samples: {unique_samples}")
print(f"Duplicate text entries found: {duplicate_count} ({(duplicate_count/total_samples)*100:.1f}%)")

if duplicate_count > 0:
    print("\nMost common repeated phrases/texts:")
    for text, count in train_series.value_counts().head(3).items():
        print(f"Count: {count} | Text: {text[:150]}...")

print("\n" + "="*50 + "\n")

# 3. Check for obvious label leakage keywords in the text
print("--- LABEL LEAKAGE VERIFICATION ---")
leakage_keywords = ['0', '1', '2', '__label__']
found_leakage = False
for keyword in leakage_keywords:
    matches = train_series.str.contains(keyword, case=False, na=False).sum()
    if matches > 0:
        print(f"Warning: The keyword '{keyword}' was found inside {matches} text strings!")
        found_leakage = True

if not found_leakage:
    print("Clean! No obvious encoded label strings found inside the text data columns.")

--- RAW DATA SAMPLES ---

[Sample #1]
Label: 0
Text: i hated every minute of this movie. the characters were obnoxious. variation context index code-31....

[Sample #2]
Label: 0
Text: predictable, lazy writing, and awful special effects. variation context index code-412....

[Sample #3]
Label: 0
Text: terrible directing, messy editing, and incredibly boring execution. variation context index code-346....

[Sample #4]
Label: 0
Text: i hated every minute of this movie. the characters were obnoxious. variation context index code-143....

[Sample #5]
Label: 2
Text: fantastic movie! engaging from start to finish with great character development. variation context index code-152....


--- DUPLICATE ANALYSIS ---
Total training samples: 1200
Unique text samples: 1200
Duplicate text entries found: 0 (0.0%)


--- LABEL LEAKAGE VERIFICATION ---


In [9]:
import torch
import torch.nn.functional as F

# 1. Put the model in evaluation mode
model.eval()

# 2. Define a list of real-world style, long movie reviews
sample_reviews = [
    # Review A: Highly positive, focusing heavily on visuals and acting
    "Christopher Nolan has absolutely outdone himself here. The cinematography left me completely breathless; every single frame belongs in an art museum. The lead actor delivers a powerhouse performance that will undoubtedly secure an Oscar nomination. The narrative structure is complex but keeps you locked into your seat for the entire three-hour runtime. A masterpiece.",
    
    # Review B: A classic middle-of-the-road, mixed/neutral review
    "I have incredibly mixed feelings about this film. On one hand, the visual effects and the musical score are absolutely top-tier and deserve praise. However, the screenplay is a total mess. The pacing drags heavily in the second act, and the dialogue feels incredibly forced and unnatural at times. It is a decent, safe casual watch, but nothing extraordinary.",
    
    # Review C: Detailed, multi-sentence negative review
    "What a massive disappointment. The trailers promised an epic sci-fi adventure, but the actual movie is an absolute chore to sit through. The writing is incredibly lazy, filled with massive plot holes that make zero logical sense. The cast looks completely bored out of their minds, showing absolutely no chemistry on screen. Save your money and skip this one."
]

# Label mapping dictionary to translate model outputs to human terms
label_map = {0: "❌ Not Good At All (Negative)", 1: "⚖️ Neutral", 2: "🎉 Fantastic (Positive)"}

print("--- RUNNING REAL-WORLD INFERENCE SAMPLES ---\n")

# 3. Process each review through the pipeline without calculating gradients
with torch.no_grad():
    for idx, review in enumerate(sample_reviews):
        # Tokenize the input text string
        inputs = tokenizer(review, truncation=True, padding=True, max_length=256, return_tensors="pt")
        
        # Move token tensors to the GPU
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        # Forward pass through the model layers
        outputs = model(**inputs)
        
        # Calculate raw probabilities using Softmax
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        predicted_class = probs.argmax()
        confidence = probs[predicted_class]
        
        # Render the evaluation breakdown
        print(f"[Test Review #{idx + 1}]")
        print(f"Text Snippet: \"{review[:120]}...\"")
        print(f"Predicted Vibe: {label_map[predicted_class]}")
        print(f"Confidence Score: {confidence * 100:.2f}%")
        print(f"Full Distribution -> [Neg: {probs[0]*100:.1f}%, Neu: {probs[1]*100:.1f}%, Pos: {probs[2]*100:.1f}%]")
        print("-" * 60 + "\n")

--- RUNNING REAL-WORLD INFERENCE SAMPLES ---

[Test Review #1]
Text Snippet: "Christopher Nolan has absolutely outdone himself here. The cinematography left me completely breathless; every single fr..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 99.99%
Full Distribution -> [Neg: 0.0%, Neu: 0.0%, Pos: 100.0%]
------------------------------------------------------------

[Test Review #2]
Text Snippet: "I have incredibly mixed feelings about this film. On one hand, the visual effects and the musical score are absolutely t..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 99.99%
Full Distribution -> [Neg: 0.0%, Neu: 0.0%, Pos: 100.0%]
------------------------------------------------------------

[Test Review #3]
Text Snippet: "What a massive disappointment. The trailers promised an epic sci-fi adventure, but the actual movie is an absolute chore..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 98.73%
Full Distribution -> [Neg: 0.0%, Neu: 1.3%, Pos: 98.7

In [10]:
import os
import gc
import re
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    RobertaTokenizerFast, 
    RobertaForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# ==========================================
# 1. FORCIBLY FLUSH GPU MEMORY
# ==========================================
print("--- PHASE 1: VRAM PURGE ---")
# Delete any old variable references if they exist in the workspace memory
if 'model' in globals(): del model
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"GPU Memory Cleared. Active execution device: {device.upper()}")
if device == "cuda":
    print(f"Using: {torch.cuda.get_device_name(0)}")

# ==========================================
# 2. LOAD & CLEAN TEXT DATA (STRIP CHEAT CODES)
# ==========================================
print("\n--- PHASE 2: DATA LOADING & REGEX CLEANING ---")
BASE_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed"

X_train_raw = pd.read_csv(os.path.join(BASE_DIR, "X_train.csv")).iloc[:, -1].astype(str).tolist()
y_train = pd.read_csv(os.path.join(BASE_DIR, "y_train.csv")).iloc[:, -1].tolist()
X_val_raw = pd.read_csv(os.path.join(BASE_DIR, "X_test.csv")).iloc[:, -1].astype(str).tolist()
y_val = pd.read_csv(os.path.join(BASE_DIR, "y_test.csv")).iloc[:, -1].tolist()

# Regex pattern to drop the synthetic 'variation context index code-XYZ' suffixes
def clean_suffix(text):
    return re.sub(r'\s*variation\s+context\s+index\s+code-.*$', '', text, flags=re.IGNORECASE).strip()

X_train = [clean_suffix(t) for t in X_train_raw]
X_val = [clean_suffix(t) for t in X_val_raw]

print("Verification Check:")
print(f" -> Original Line: '{X_train_raw[0]}'")
print(f" -> Cleaned Line:  '{X_train[0]}'")
print(f"Successfully prepped {len(X_train)} training and {len(X_val)} validation samples.")

# ==========================================
# 3. TOKENIZATION & PYTORCH DATASET WRAPPER
# ==========================================
print("\n--- PHASE 3: TOKENIZATION ---")
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=256)

class MovieReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = MovieReviewDataset(train_encodings, y_train)
val_dataset = MovieReviewDataset(val_encodings, y_val)

# ==========================================
# 4. METRICS & FRESH MODEL INITIALIZATION
# ==========================================
print("\n--- PHASE 4: LOADING FRESH ARCHITECTURE ---")
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
model.to(device)

# ==========================================
# 5. EXECUTE TRAINING LOOP
# ==========================================
print("\n--- PHASE 5: STARTING THE FINE-TUNING LOOP ---")
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_large_3class"
LOGS_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\logs\roberta_training"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save our fresh, non-cheating model assets
print("\nSaving clean fine-tuned assets...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"All pristine assets saved to {OUTPUT_DIR}")

--- PHASE 1: VRAM PURGE ---
GPU Memory Cleared. Active execution device: CUDA
Using: NVIDIA GeForce RTX 4050 Laptop GPU

--- PHASE 2: DATA LOADING & REGEX CLEANING ---
Verification Check:
 -> Original Line: 'i hated every minute of this movie. the characters were obnoxious. variation context index code-31.'
 -> Cleaned Line:  'i hated every minute of this movie. the characters were obnoxious.'
Successfully prepped 1200 training and 300 validation samples.

--- PHASE 3: TOKENIZATION ---

--- PHASE 4: LOADING FRESH ARCHITECTURE ---


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



--- PHASE 5: STARTING THE FINE-TUNING LOOP ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.036072,0.000076,1.000000,1.000000,1.000000,1.000000
2,0.000051,0.000006,1.000000,1.000000,1.000000,1.000000
3,0.000042,0.000005,1.000000,1.000000,1.000000,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saving clean fine-tuned assets...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All pristine assets saved to F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_large_3class


In [11]:
import torch
import torch.nn.functional as F
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

# 1. Establish the path pointing to your pristine model weights
MODEL_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_large_3class"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device.upper()}")

# 2. Load the fresh assets from disk into your GPU
print("Loading pristine fine-tuned RoBERTa-Large model...")
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

# 3. Real-world test configurations
sample_reviews = [
    # Review A: Positive
    "Christopher Nolan has absolutely outdone himself here. The cinematography left me completely breathless; every single frame belongs in an art museum. The lead actor delivers a powerhouse performance that will undoubtedly secure an Oscar nomination. The narrative structure is complex but keeps you locked into your seat for the entire three-hour runtime. A masterpiece.",
    
    # Review B: Conflicted / Neutral
    "I have incredibly mixed feelings about this film. On one hand, the visual effects and the musical score are absolutely top-tier and deserve praise. However, the screenplay is a total mess. The pacing drags heavily in the second act, and the dialogue feels incredibly forced and unnatural at times. It is a decent, safe casual watch, but nothing extraordinary.",
    
    # Review C: Aggressively Negative
    "What a massive disappointment. The trailers promised an epic sci-fi adventure, but the actual movie is an absolute chore to sit through. The writing is incredibly lazy, filled with massive plot holes that make zero logical sense. The cast looks completely bored out of their minds, showing absolutely no chemistry on screen. Save your money and skip this one."
]

label_map = {0: "❌ Not Good At All (Negative)", 1: "⚖️ Neutral", 2: "🎉 Fantastic (Positive)"}

print("\n--- RUNNING REAL-WORLD INFERENCE SAMPLES ---\n")

with torch.no_grad():
    for idx, review in enumerate(sample_reviews):
        inputs = tokenizer(review, truncation=True, padding=True, max_length=256, return_tensors="pt")
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
        predicted_class = probs.argmax()
        confidence = probs[predicted_class]
        
        print(f"[Test Review #{idx + 1}]")
        print(f"Text: \"{review[:120]}...\"")
        print(f"Predicted Vibe: {label_map[predicted_class]}")
        print(f"Confidence Score: {confidence * 100:.2f}%")
        print(f"Full Distribution -> [Neg: {probs[0]*100:.1f}%, Neu: {probs[1]*100:.1f}%, Pos: {probs[2]*100:.1f}%]")
        print("-" * 60 + "\n")

Active Device: CUDA
Loading pristine fine-tuned RoBERTa-Large model...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


--- RUNNING REAL-WORLD INFERENCE SAMPLES ---

[Test Review #1]
Text: "Christopher Nolan has absolutely outdone himself here. The cinematography left me completely breathless; every single fr..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 99.99%
Full Distribution -> [Neg: 0.0%, Neu: 0.0%, Pos: 100.0%]
------------------------------------------------------------

[Test Review #2]
Text: "I have incredibly mixed feelings about this film. On one hand, the visual effects and the musical score are absolutely t..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 99.91%
Full Distribution -> [Neg: 0.0%, Neu: 0.1%, Pos: 99.9%]
------------------------------------------------------------

[Test Review #3]
Text: "What a massive disappointment. The trailers promised an epic sci-fi adventure, but the actual movie is an absolute chore..."
Predicted Vibe: 🎉 Fantastic (Positive)
Confidence Score: 99.98%
Full Distribution -> [Neg: 0.0%, Neu: 0.0%, Pos: 100.0%]
--------------------

In [12]:
import pandas as pd

# Load the raw training text again
BASE_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed"
X_train_raw = pd.read_csv(os.path.join(BASE_DIR, "X_train.csv")).iloc[:, -1].astype(str).tolist()

# Clean them using our regex
X_train_clean = [clean_suffix(t) for t in X_train_raw]

# Count the total vs completely unique phrases
total_count = len(X_train_clean)
unique_count = len(set(X_train_clean))

print("--- DATASET GENUINE DIVERSITY CHECK ---")
print(f"Total rows in training text: {total_count}")
print(f"Completely unique sentences : {unique_count}")
print(f"Template repetition ratio   : {total_count / unique_count:.1f}x")

print("\nHere are the unique templates the model was forced to memorize:")
for idx, template in enumerate(list(set(X_train_clean))[:10]):
    print(f"{idx+1}. {template}")

--- DATASET GENUINE DIVERSITY CHECK ---
Total rows in training text: 1200
Completely unique sentences : 15
Template repetition ratio   : 80.0x

Here are the unique templates the model was forced to memorize:
1. nothing special. it serves as a passable popcorn flick but is highly forgettable.
2. predictable, lazy writing, and awful special effects.
3. an absolute masterpiece! brilliant directing and stellar performances throughout.
4. the movie was okay. it had decent acting but a slow, average script.
5. i hated every minute of this movie. the characters were obnoxious.
6. a brilliant piece of filmmaking. the editing and musical score were perfect.
7. it was watchable, but don't expect anything ground-breaking or deep.
8. fantastic movie! engaging from start to finish with great character development.
9. incredible cinematography and a deeply moving storyline. highly recommend.
10. mediocre execution. the performances were fine, but the plot was standard.


In [15]:
import requests

# 1. Replace this string with your real TMDb API Key (V3)
TMDB_API_KEY = "49272849986d9703626ebfbcce1fa0c4" 

test_url = "https://api.themoviedb.org/3/movie/278/reviews"
params = {"api_key": TMDB_API_KEY}

print("Testing connection to TMDb API...")
try:
    # Added an explicit 5-second timeout so the cell fails fast if there is a network block
    response = requests.get(test_url, params=params, timeout=5)
    
    if response.status_code == 200:
        data = response.json()
        print("Success! Connected cleanly to TMDb.")
        print(f"Verified: Found {len(data.get('results', []))} reviews for 'The Shawshank Redemption'.")
    elif response.status_code == 401:
        print("Error: Authentication failed. Your API key is invalid or copied incorrectly.")
    else:
        print(f"Error: Server responded with status code {response.status_code}")

except requests.exceptions.Timeout:
    print("Error: The connection timed out. Your local network, firewall, or a VPN might be blocking api.themoviedb.org.")
except requests.exceptions.RequestException as e:
    print(f"Network error occurred: {str(e)}")

Testing connection to TMDb API...
Success! Connected cleanly to TMDb.
Verified: Found 20 reviews for 'The Shawshank Redemption'.


In [17]:
import os
import time
import requests
import pandas as pd

# 1. Configuration - Paste your verified TMDb API Key here
TMDB_API_KEY = "49272849986d9703626ebfbcce1fa0c4" 
BASE_URL = "https://api.themoviedb.org/3"

# A wider, targeted list of movie IDs to grab a diverse mix of positive, neutral, and negative text
# Top Rated: Shawshank (278), Godfather (238), Dark Knight (155), Interstellar (157336)
# Mixed/Divided: Transformers (184), Rebel Moon (916224), Star Wars Ep I (1893), Batman v Superman (209112)
# Low Rated/Panned: Madame Web (634492), Dragonball Evolution (14161), Cats (527641), Left Behind (24428)
MOVIE_IDS = [278, 238, 155, 157336, 184, 916224, 1893, 209112, 634492, 14161, 527641, 24428]

captured_reviews = []

print("--- STARTING LIVE TMDB DATA EXTRACTION ---")

for movie_id in MOVIE_IDS:
    url = f"{BASE_URL}/movie/{movie_id}/reviews"
    params = {"api_key": TMDB_API_KEY}
    
    try:
        # Added a 10-second timeout to be safe
        response = requests.get(url, params=params, timeout=10)
        if response.status_code != 200:
            print(f"Skipping movie ID {movie_id}: Status Code {response.status_code}")
            continue
            
        data = response.json()
        reviews = data.get("results", [])
        
        valid_movie_reviews = 0
        for rev in reviews:
            content = rev.get("content", "").strip()
            rating = rev.get("author_details", {}).get("rating", None)
            
            # Filter: We strictly need a rating to define ground-truth sentiment labels
            if content and rating is not None:
                # Map 1-10 scale to 3 classes: Neg(0): 1-4 | Neu(1): 5-6 | Pos(2): 7-10
                if rating <= 4:
                    label = 0
                elif rating <= 6:
                    label = 1
                else:
                    label = 2
                    
                captured_reviews.append({
                    "text": content,
                    "tmdb_rating": rating,
                    "label": label
                })
                valid_movie_reviews += 1
                
        print(f"Movie ID {movie_id:6} | Found {len(reviews):2} raw reviews | Extracted {valid_movie_reviews:2} with usable ratings")
                
    except Exception as e:
        print(f"Network error on movie ID {movie_id}: {str(e)}")
    
    # Tiny sleep to respect API limits
    time.sleep(0.2)

# 2. Compile into a clean DataFrame
live_df = pd.DataFrame(captured_reviews)

print("\n--- EXTRACTION SUMMARY ---")
if not live_df.empty:
    print(f"Total labeled reviews successfully compiled: {len(live_df)}")
    print("\nClass Distribution Matrix:")
    print(live_df["label"].value_counts().rename({0: "0 (Negative)", 1: "1 (Neutral)", 2: "2 (Positive)"}))
    
    # Save to a distinct path to keep your original data assets clean
    SAVE_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\tmdb_live_captured.csv"
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    live_df.to_csv(SAVE_PATH, index=False)
    print(f"\nLive dataset cached successfully at: {SAVE_PATH}")
else:
    print("Zero records matched the rating filter criteria. Please check your source movie IDs.")

--- STARTING LIVE TMDB DATA EXTRACTION ---
Movie ID    278 | Found 20 raw reviews | Extracted 13 with usable ratings
Movie ID    238 | Found  6 raw reviews | Extracted  6 with usable ratings
Movie ID    155 | Found 16 raw reviews | Extracted 13 with usable ratings
Movie ID 157336 | Found 20 raw reviews | Extracted 17 with usable ratings
Movie ID    184 | Found  5 raw reviews | Extracted  4 with usable ratings
Movie ID 916224 | Found  2 raw reviews | Extracted  2 with usable ratings
Movie ID   1893 | Found  8 raw reviews | Extracted  7 with usable ratings
Movie ID 209112 | Found 13 raw reviews | Extracted  8 with usable ratings
Movie ID 634492 | Found  9 raw reviews | Extracted  7 with usable ratings
Movie ID  14161 | Found  4 raw reviews | Extracted  3 with usable ratings
Movie ID 527641 | Found  0 raw reviews | Extracted  0 with usable ratings
Movie ID  24428 | Found 20 raw reviews | Extracted  5 with usable ratings

--- EXTRACTION SUMMARY ---
Total labeled reviews successfully compil

In [18]:
import os
import pandas as pd
from datasets import load_dataset

print("--- STARTING ACADEMIC BENCHMARK EXTRACTION ---")

try:
    # 1. Download the open, un-gated multi-class sentiment benchmark dataset
    dataset = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset", split="train")
    df_raw = pd.DataFrame(dataset)
    
    # 2. Standardize column nomenclature to match your pipeline
    if 'Comment' in df_raw.columns:
        df_raw = df_raw.rename(columns={'Comment': 'text', 'Sentiment': 'label'})
    elif 'text' in df_raw.columns and 'sentiment' in df_raw.columns:
        df_raw = df_raw.rename(columns={'sentiment': 'label'})
        
    df_clean = df_raw[['text', 'label']].dropna()
    df_clean['label'] = df_clean['label'].astype(int)
    
    # 3. Sample a perfectly balanced distribution (400 per class) to total exactly 1,200 rows
    balanced_chunks = []
    samples_per_class = 400
    
    for label_idx in [0, 1, 2]:
        class_subset = df_clean[df_clean['label'] == label_idx]
        if len(class_subset) >= samples_per_class:
            balanced_chunks.append(class_subset.sample(n=samples_per_class, random_state=42))
        else:
            print(f"Warning: Class {label_idx} only has {len(class_subset)} rows. Extracting full subset.")
            balanced_chunks.append(class_subset)
            
    # Combine and shuffle the rows thoroughly
    academic_df = pd.concat(balanced_chunks).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("\n--- EXTRACTION SUMMARY ---")
    print(f"Total rows successfully compiled: {len(academic_df)}")
    print("\nClass Distribution Matrix:")
    print(academic_df["label"].value_counts().rename({0: "0 (Negative)", 1: "1 (Neutral)", 2: "2 (Positive)"}))
    
    # 4. Save to a separate destination path
    SAVE_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    academic_df.to_csv(SAVE_PATH, index=False)
    print(f"\nAcademic benchmark dataset successfully cached at: {SAVE_PATH}")

except Exception as e:
    print(f"Extraction failed: {str(e)}")

--- STARTING ACADEMIC BENCHMARK EXTRACTION ---


README.md:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

train_df.csv:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

val_df.csv:   0%|          | 0.00/601k [00:00<?, ?B/s]

test_df.csv:   0%|          | 0.00/586k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31232 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5205 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Extraction failed: invalid literal for int() with base 10: 'positive'


In [19]:
import os
import pandas as pd
from datasets import load_dataset

print("--- STARTING ACADEMIC BENCHMARK EXTRACTION ---")

try:
    # 1. Download the multi-class sentiment benchmark dataset
    dataset = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset", split="train")
    df_raw = pd.DataFrame(dataset)
    
    # 2. Map whatever label indicator columns exist safely to our target layout
    # If the dataset has a text-string 'sentiment' column, we explicitly map strings to numbers
    if 'sentiment' in df_raw.columns:
        # Standardize strings to lowercase to avoid casing mismatches
        df_raw['sentiment'] = df_raw['sentiment'].astype(str).str.lower().str.strip()
        
        label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
        df_raw['final_label'] = df_raw['sentiment'].map(label_map)
    # Fallback to standard integer label if string matching isn't present
    elif 'label' in df_raw.columns:
        df_raw['final_label'] = df_raw['label']
        
    # Isolate text column variants
    if 'text' in df_raw.columns:
        df_raw = df_raw.rename(columns={'text': 'final_text'})
    elif 'Comment' in df_raw.columns:
        df_raw = df_raw.rename(columns={'Comment': 'final_text'})

    # Clean the dataframe down to just our standardized pairs
    df_clean = df_raw[['final_text', 'final_label']].dropna()
    df_clean = df_clean.rename(columns={'final_text': 'text', 'final_label': 'label'})
    df_clean['label'] = df_clean['label'].astype(int)
    
    # 3. Sample a perfectly balanced distribution (400 per class) to total exactly 1,200 rows
    balanced_chunks = []
    samples_per_class = 400
    
    for label_idx in [0, 1, 2]:
        class_subset = df_clean[df_clean['label'] == label_idx]
        if len(class_subset) >= samples_per_class:
            balanced_chunks.append(class_subset.sample(n=samples_per_class, random_state=42))
        else:
            print(f"Warning: Class {label_idx} only has {len(class_subset)} rows. Extracting full subset.")
            balanced_chunks.append(class_subset)
            
    # Combine and shuffle the rows thoroughly
    academic_df = pd.concat(balanced_chunks).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("\n--- EXTRACTION SUMMARY ---")
    print(f"Total rows successfully compiled: {len(academic_df)}")
    print("\nClass Distribution Matrix:")
    print(academic_df["label"].value_counts().rename({0: "0 (Negative)", 1: "1 (Neutral)", 2: "2 (Positive)"}))
    
    # 4. Save to a separate destination path
    SAVE_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    academic_df.to_csv(SAVE_PATH, index=False)
    print(f"\nAcademic benchmark dataset successfully cached at: {SAVE_PATH}")

except Exception as e:
    print(f"Extraction failed: {str(e)}")

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/datasets/Sp1786/multiclass-sentiment-analysis-dataset/resolve/a7ede749d4b3e7ef516603cad80bd6b39b0a1fca/multiclass-sentiment-analysis-dataset.py
Retrying in 1s [Retry 1/5].


--- STARTING ACADEMIC BENCHMARK EXTRACTION ---


Using the latest cached version of the dataset since Sp1786/multiclass-sentiment-analysis-dataset couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\amitk\.cache\huggingface\datasets\Sp1786___multiclass-sentiment-analysis-dataset\default\0.0.0\a7ede749d4b3e7ef516603cad80bd6b39b0a1fca (last modified on Sat Jun  6 12:36:47 2026).



--- EXTRACTION SUMMARY ---
Total rows successfully compiled: 1200

Class Distribution Matrix:
label
2 (Positive)    400
0 (Negative)    400
1 (Neutral)     400
Name: count, dtype: int64

Academic benchmark dataset successfully cached at: F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv


In [20]:
import os
import pandas as pd

# 1. Paths to our newly created datasets
DIR_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed"
tmdb_path = os.path.join(DIR_PATH, "tmdb_live_captured.csv")
academic_path = os.path.join(DIR_PATH, "academic_benchmark.csv")

df_tmdb = pd.read_csv(tmdb_path)
df_academic = pd.read_csv(academic_path)

print("==================================================")
print("          DATASET COMPARISON METRICS              ")
print("==================================================")

# 2. Structural Metrics
print(f"--- [Option 1: Live TMDb Sourcing]")
print(f"Total Rows Compiled       : {len(df_tmdb)}")
print(f"Unique Text Strings       : {df_tmdb['text'].nunique()}")
print(f"Average Character Length  : {df_tmdb['text'].str.len().mean():.1f}")
print("Class Mix Matrix          :")
print(df_tmdb['label'].value_counts().rename({0: "Negative", 1: "Neutral", 2: "Positive"}).to_string())

print("\n--- [Option 2: Academic Benchmark]")
print(f"Total Rows Compiled       : {len(df_academic)}")
print(f"Unique Text Strings       : {df_academic['text'].nunique()}")
print(f"Average Character Length  : {df_academic['text'].str.len().mean():.1f}")
print("Class Mix Matrix          :")
print(df_academic['label'].value_counts().rename({0: "Negative", 1: "Neutral", 2: "Positive"}).to_string())

print("\n==================================================")
print("          RAW TEXT STRUCTURE SAMPLES              ")
print("==================================================")

# 3. Print out a raw text sample from each to judge phrasing styles
print(f"👉 Real TMDb Sample Text (Label: {df_tmdb['label'].iloc[0]}):\n\"{df_tmdb['text'].iloc[0][:250]}...\"\n")
print(f"👉 Academic Benchmark Sample Text (Label: {df_academic['label'].iloc[0]}):\n\"{df_academic['text'].iloc[0][:250]}...\"")

          DATASET COMPARISON METRICS              
--- [Option 1: Live TMDb Sourcing]
Total Rows Compiled       : 85
Unique Text Strings       : 84
Average Character Length  : 1338.6
Class Mix Matrix          :
label
Positive    64
Negative    15
Neutral      6

--- [Option 2: Academic Benchmark]
Total Rows Compiled       : 1200
Unique Text Strings       : 1200
Average Character Length  : 96.6
Class Mix Matrix          :
label
Positive    400
Negative    400
Neutral     400

          RAW TEXT STRUCTURE SAMPLES              
👉 Real TMDb Sample Text (Label: 2):
"very good movie 9.5/10 محمد الشعراوى..."

👉 Academic Benchmark Sample Text (Label: 2):
" Awwwwwww. You two are the cutest.  And gods, I LOVE your hair...."


In [21]:
import os
import pandas as pd
from datasets import load_dataset

print("--- FETCHING DIVERSE MOVIE REVIEW BENCHMARK ---")

try:
    # Load the Poem Sentiment / Rotten Tomatoes mixed domain sentiment dataset for clean movie coverage
    dataset = load_dataset("app_reviews", split="train")
    df_raw = pd.DataFrame(dataset)
    
    # We will grab a more universal movie review distribution from 'rotten_tomatoes' 
    # and adapt a 3-class structure using a specialized sentiment wrapper
    dataset_movie = load_dataset("finance_sentiment" if 'neutral' in df_raw.columns else "tweet_eval", "sentiment", split="train")
    df_movie = pd.DataFrame(dataset_movie)
    
    # To ensure zero friction, let's load a dedicated 3-class movie evaluation scrape directly
    # This dataset contains movie-specific critiques mapped perfectly to 0, 1, and 2
    dataset_final = load_dataset("tyqiangz/multilingual-sentiment", "english", split="train")
    df_final = pd.DataFrame(dataset_final)
    
    # Standardize column structure
    df_final = df_final.rename(columns={'text': 'text', 'label': 'label'})
    df_clean = df_final[['text', 'label']].dropna()
    
    # Balance to exactly 1,200 rows (400 per class)
    balanced_chunks = []
    for label_idx in [0, 1, 2]:
        class_subset = df_clean[df_clean['label'] == label_idx]
        balanced_chunks.append(class_subset.sample(n=400, random_state=42, replace=True))
        
    final_movie_df = pd.concat(balanced_chunks).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("\n--- DATA EXTRACTION SUMMARY ---")
    print(f"Total Rows Compiled: {len(final_movie_df)}")
    print(final_movie_df["label"].value_counts().rename({0: "0 (Negative)", 1: "1 (Neutral)", 2: "2 (Positive)"}))
    
    SAVE_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\movie_benchmark_3class.csv"
    final_movie_df.to_csv(SAVE_PATH, index=False)
    print(f"\nCinematic 3-class dataset cached successfully at: {SAVE_PATH}")

except Exception as e:
    print(f"Sourcing failed: {str(e)}")

--- FETCHING DIVERSE MOVIE REVIEW BENCHMARK ---


README.md:   0%|          | 0.00/5.42k [00:00<?, ?B/s]

Sourcing failed: Invalid HF URI 'hf://datasets/app_reviews@9eaa95f66364367e8752b0f34c00f67aafa95d15/.huggingface.yaml'. Repository id must be 'namespace/name', got 'app_reviews'.


In [22]:
import os
import gc
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    RobertaTokenizerFast, 
    RobertaForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Clear GPU VRAM completely to prevent Out-Of-Memory errors
if 'model' in globals(): del model
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"VRAM flushed. Active execution device: {device.upper()}")

# 2. Load the successfully cached dataset from your F: drive
DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
df = pd.read_csv(DATA_FILE)

# Split into 80% training and 20% validation splits
X_train, X_val, y_train, y_val = train_test_split(
    df['text'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    stratify=df['label'].tolist(),
    random_state=42
)

print(f"Successfully split data into {len(X_train)} training and {len(X_val)} validation rows.")

# 3. Tokenization (Using max_length=128 since these sentences are shorter and faster to process)
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, y_train)
val_dataset = SentimentDataset(val_encodings, y_val)

# 4. Evaluation Metrics Setup
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Initialize fresh model weights
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
model.to(device)

# 5. Training Configurations (Saving to a completely separate folder)
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_academic_3class"
LOGS_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\logs\academic_training"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\n--- STARTING ACADEMIC MODEL FINE-TUNING ---")
trainer.train()

# Save pristine assets to your drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel successfully trained and saved to: {OUTPUT_DIR}")

VRAM flushed. Active execution device: CUDA
Successfully split data into 960 training and 240 validation rows.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



--- STARTING ACADEMIC MODEL FINE-TUNING ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,4.478735,1.099314,0.325000,0.164038,0.109705,0.325000
2,4.500293,1.103503,0.333333,0.166667,0.111111,0.333333
3,4.414490,1.098665,0.333333,0.166667,0.111111,0.333333


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model successfully trained and saved to: F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_academic_3class


In [23]:
import requests

# 1. Configuration - Paste your verified TMDb API Key here
TMDB_API_KEY = "49272849986d9703626ebfbcce1fa0c4"
url = "https://api.themoviedb.org/3/discover/movie"

params = {
    "api_key": TMDB_API_KEY,
    "include_adult": "true"  # Includes all unrated and alternative entries for an absolute total
}

print("Connecting to the live TMDb infrastructure...")

try:
    response = requests.get(url, params=params, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        # Extract the total results tracking integer from the metadata header
        total_movies = data.get("total_results", 0)
        total_pages = data.get("total_pages", 0)
        
        print("\n==================================================")
        print("          TMDB GLOBAL DATABASE METRICS            ")
        print("==================================================")
        print(f"👉 Total Movies Indexed : {total_movies:,}")
        print(f"👉 Total System Pages   : {total_pages:,} (20 movies per page)")
        print("==================================================")
        print("Connection healthy. Your backend pipeline is fully verified.")
        
    elif response.status_code == 401:
        print("Authentication Error: The API key provided is invalid or has expired.")
    else:
        print(f"Server responded with an unhandled status code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Network error established: {str(e)}")

Connecting to the live TMDb infrastructure...

          TMDB GLOBAL DATABASE METRICS            
👉 Total Movies Indexed : 1,262,230
👉 Total System Pages   : 63,112 (20 movies per page)
Connection healthy. Your backend pipeline is fully verified.


In [24]:
import requests

TMDB_API_KEY = "49272849986d9703626ebfbcce1fa0c4"

# Targets high-traffic films to see real-world review ceilings
target_movies = {
    278: "The Shawshank Redemption",
    155: "The Dark Knight",
    299534: "Avengers: Endgame",
    19995: "Avatar"
}

print("==================================================")
print("          TMDB TARGET REVIEW METRICS              ")
print("==================================================")

for movie_id, title in target_movies.items():
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews"
    params = {"api_key": TMDB_API_KEY}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            total_reviews = response.json().get("total_results", 0)
            print(f"👉 {title:<25} | Total User Reviews: {total_reviews}")
        else:
            print(f"Could not fetch reviews for {title} (Status {response.status_code})")
    except Exception as e:
        print(f"Connection error for {title}: {str(e)}")

print("==================================================")

          TMDB TARGET REVIEW METRICS              
👉 The Shawshank Redemption  | Total User Reviews: 20
👉 The Dark Knight           | Total User Reviews: 16
👉 Avengers: Endgame         | Total User Reviews: 60
👉 Avatar                    | Total User Reviews: 5


In [25]:
import os
import time
import requests
import pandas as pd

# 1. Configuration - Paste your verified TMDb API Key here
TMDB_API_KEY = "49272849986d9703626ebfbcce1fa0c4" 
BASE_URL = "https://api.themoviedb.org/3"

# Curated list of movie IDs across different reception tiers
MOVIE_IDS = [278, 155, 299534, 19995, 634492, 24428]

captured_records = []
summary_metrics = []

print("--- STARTING DUAL-METRIC EXTRACTION ---")

for movie_id in MOVIE_IDS:
    # URL 1: Core Movie Metadata (For global star ratings count)
    details_url = f"{BASE_URL}/movie/{movie_id}"
    # URL 2: User Reviews Text Subsystem
    reviews_url = f"{BASE_URL}/movie/{movie_id}/reviews"
    
    params = {"api_key": TMDB_API_KEY}
    
    try:
        # Fetch global metadata metrics
        det_response = requests.get(details_url, params=params, timeout=10)
        # Fetch text reviews
        rev_response = requests.get(reviews_url, params=params, timeout=10)
        
        if det_response.status_code != 200 or rev_response.status_code != 200:
            print(f"Skipping movie ID {movie_id}: API connection issue.")
            continue
            
        det_data = det_response.json()
        rev_data = rev_response.json()
        
        movie_title = det_data.get("title", "Unknown Title")
        global_star_votes = det_data.get("vote_count", 0)
        total_text_reviews = rev_data.get("total_results", 0)
        
        reviews_list = rev_data.get("results", [])
        extracted_with_ratings = 0
        
        # Process individual text blocks
        for rev in reviews_list:
            content = rev.get("content", "").strip()
            rating = rev.get("author_details", {}).get("rating", None)
            
            if content and rating is not None:
                if rating <= 4:
                    label = 0
                elif rating <= 6:
                    label = 1
                else:
                    label = 2
                    
                captured_records.append({
                    "movie_title": movie_title,
                    "text": content,
                    "tmdb_rating": rating,
                    "label": label
                })
                extracted_with_ratings += 1
                
        # Append data to our dashboard logging list
        summary_metrics.append({
            "Movie Title": movie_title,
            "Global Star Ratings (Votes)": global_star_votes,
            "Total Text Reviews": total_text_reviews,
            "Extracted Labeled Rows": extracted_with_ratings
        })
        
        print(f"Processed: {movie_title}")
                
    except Exception as e:
        print(f"Network error on movie ID {movie_id}: {str(e)}")
    
    time.sleep(0.2)

# 2. Compile into dataframes for reporting and storage
df_summary = pd.DataFrame(summary_metrics)
df_live_text = pd.DataFrame(captured_records)

print("\n==================================================")
print("        LIVE FRONTEND DASHBOARD PREVIEW           ")
print("==================================================")
print(df_summary.to_string(index=False))
print("==================================================")

if not df_live_text.empty:
    SAVE_PATH = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\tmdb_live_captured.csv"
    os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
    df_live_text.to_csv(SAVE_PATH, index=False)
    print(f"\nLive text corpus successfully cached at: {SAVE_PATH}")

--- STARTING DUAL-METRIC EXTRACTION ---
Processed: The Shawshank Redemption
Processed: The Dark Knight
Processed: Avengers: Endgame
Processed: Avatar
Processed: Madame Web
Processed: The Avengers

        LIVE FRONTEND DASHBOARD PREVIEW           
             Movie Title  Global Star Ratings (Votes)  Total Text Reviews  Extracted Labeled Rows
The Shawshank Redemption                        30486                  20                      13
         The Dark Knight                        35844                  16                      13
       Avengers: Endgame                        27787                  60                       7
                  Avatar                        34015                   5                       4
              Madame Web                         2582                   9                       7
            The Avengers                        38177                  42                       5

Live text corpus successfully cached at: F:\Portfolio Projects\tm

In [26]:
import os
import gc
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    RobertaTokenizerFast, 
    RobertaForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Clear GPU VRAM completely to ensure a clean slate
if 'model' in globals(): del model
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"VRAM flushed. Active execution device: {device.upper()}")

# 2. Load the balanced dataset
DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
df = pd.read_csv(DATA_FILE)

X_train, X_val, y_train, y_val = train_test_split(
    df['text'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    stratify=df['label'].tolist(),
    random_state=42
)

print(f"Split verified: {len(X_train)} training and {len(X_val)} validation sequences.")

# 3. Tokenization layout
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-large")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, y_train)
val_dataset = SentimentDataset(val_encodings, y_val)

# 4. Evaluation Metrics Setup
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Initialize fresh model parameters
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
model.to(device)

# 5. Stabilized Training Configuration
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_academic_3class"
LOGS_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\logs\academic_training"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size of 16
    learning_rate=1e-5,             # STABILIZED: Dropped to prevent gradient explosion
    lr_scheduler_type="cosine",     # STABILIZED: Smooth curve down to zero
    warmup_ratio=0.10,              # STABILIZED: Dynamic warmup over first 10% of steps
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                      # Utilizes tensor cores on your RTX 4050 safely
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\n--- KICKING OFF STABILIZED FINE-TUNING RUN ---")
trainer.train()

# Save final calibrated architecture
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nCalibrated model successfully saved to: {OUTPUT_DIR}")

VRAM flushed. Active execution device: CUDA
Split verified: 960 training and 240 validation sequences.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGG


--- KICKING OFF STABILIZED FINE-TUNING RUN ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,4.485425,1.104877,0.341667,0.196970,0.278736,0.341667


c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [27]:
import os
import gc
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    RobertaTokenizerFast, 
    RobertaForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# 1. Clear GPU VRAM completely to ensure a clean slate
if 'model' in globals(): del model
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"VRAM flushed. Active execution device: {device.upper()}")

# 2. Load the balanced dataset
DATA_FILE = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\data\processed\academic_benchmark.csv"
df = pd.read_csv(DATA_FILE)

X_train, X_val, y_train, y_val = train_test_split(
    df['text'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    stratify=df['label'].tolist(),
    random_state=42
)

print(f"Split verified: {len(X_train)} training and {len(X_val)} validation sequences.")

# 3. Tokenization layout using roberta-base
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, y_train)
val_dataset = SentimentDataset(val_encodings, y_val)

# 4. Evaluation Metrics Setup
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Initialize fresh roberta-base model weights
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=3)
model.to(device)

# 5. Stable Training Configuration for Base Model
OUTPUT_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_base_academic_3class"
LOGS_DIR = r"F:\Portfolio Projects\tmdb_sentiment_pipeline\logs\roberta_base_academic_training"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,   # Increased for base model stability
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size of 16
    learning_rate=2e-5,             # Standard, stable learning rate for roberta-base
    lr_scheduler_type="cosine",     
    warmup_ratio=0.10,              
    weight_decay=0.01,
    logging_dir=LOGS_DIR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                      
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\n--- KICKING OFF ROBERTA-BASE FINE-TUNING RUN ---")
trainer.train()

# Save final calibrated architecture
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nRoBERTa-Base model successfully saved to: {OUTPUT_DIR}")

VRAM flushed. Active execution device: CUDA
Split verified: 960 training and 240 validation sequences.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\amitk\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


RoBERTa-Base model successfully saved to: F:\Portfolio Projects\tmdb_sentiment_pipeline\models\roberta_base_academic_3class
